# Similar Movies Debug Runner

Run the setup cell once, set `tmdb_ids`, then execute the result cell to inspect the standalone similar-movies flow with lane labels.

## Cell 1 - Setup

In [1]:
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find project root containing pyproject.toml")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from db.postgres import fetch_movie_cards, pool as postgres_pool
from search_v2.similar_movies import run_similar_movies_for_ids

if postgres_pool._closed:
    await postgres_pool.open()

print(f"Project root: {PROJECT_ROOT}")
print("Postgres pool open")

/Users/michaelkeohane/Documents/movie-finder-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /Users/michaelkeohane/Documents/movie-finder-rag
Postgres pool open


## Cell 2 - Anchors

## Cell 3 - Results

In [2]:
tmdb_ids = [155]
limit = 25

def year_from_release_ts(release_ts: int | None) -> int | None:
    if release_ts is None:
        return None
    return datetime.fromtimestamp(release_ts, tz=timezone.utc).year


result = await run_similar_movies_for_ids(tmdb_ids, limit=limit)

cards = await fetch_movie_cards([item.movie_id for item in result.ranked])
cards_by_id = {card["movie_id"]: card for card in cards}

rows = []
for rank, item in enumerate(result.ranked, start=1):
    card = cards_by_id.get(item.movie_id, {})
    lane_scores = item.evidence.lane_scores
    rows.append(
        {
            "rank": rank,
            "title": card.get("title", "<missing card>"),
            "year": year_from_release_ts(card.get("release_ts")),
            "tmdb_id": item.movie_id,
            "score": round(item.score, 4),
            "dominant_lane": item.evidence.dominant_lane,
            "lanes": ", ".join(item.evidence.candidate_sources),
            "shape": round(lane_scores.get("shape", 0.0), 4),
            "director": round(lane_scores.get("director", 0.0), 4),
            "franchise": round(lane_scores.get("franchise", 0.0), 4),
            "studio": round(lane_scores.get("studio", 0.0), 4),
            "source": round(lane_scores.get("source", 0.0), 4),
            "quality": round(lane_scores.get("quality", 0.0), 4),
        }
    )

print("anchors:", result.anchor_movie_ids)
print("active_anchor_types:", result.active_anchor_types)
print("lane_weights:", result.debug.normalized_lane_weights)
print("vector_space_weights:", result.debug.vector_space_weights)
display(pd.DataFrame(rows))

anchors: [155]
active_anchor_types: ['standard_shape', 'prestige', 'franchise_dominant', 'source_material', 'director_signature']
lane_weights: {'shape': 0.34426229508196726, 'franchise': 0.2459016393442623, 'source': 0.09836065573770492, 'quality': 0.18032786885245902, 'format': 0.03278688524590164, 'themes': 0.09836065573770492, 'cast': 0.0, 'specific_award': 0.0, 'studio': 0.06}
vector_space_weights: {'anchor': 0.09278350515463918, 'plot_events': 0.051546391752577324, 'plot_analysis': 0.2061855670103093, 'viewer_experience': 0.2061855670103093, 'watch_context': 0.15463917525773196, 'narrative_techniques': 0.11340206185567012, 'production': 0.061855670103092786, 'reception': 0.11340206185567012}


,rank,title,year,tmdb_id,score,dominant_lane,lanes,shape,director,franchise,studio,source,quality
0,1,The Dark Knight Rises,2012,49026,1.2071,shape,"shape, director, franchise, source, quality, f...",0.8862,1.0,1.0,0.0,0.3521,0.5660
1,2,Oppenheimer,2023,872585,0.4797,quality,"shape, director, quality, format, themes",0.3275,1.0,0.0,0.0,0.0000,0.8720
2,3,Captain America: Civil War,2016,271110,0.4211,shape,"shape, source, quality, format, themes",0.6056,0.0,0.0,0.0,0.3521,0.3980
3,4,L.A. Confidential,1997,2118,0.4166,quality,"shape, quality, format, themes",0.4393,0.0,0.0,0.0,0.0000,0.8960
4,5,Batman Begins,2005,272,1.0916,shape,"shape, director, franchise, source, quality, f...",0.7717,1.0,1.0,0.0,0.3521,0.3500
5,6,Man of Steel,2013,49521,0.4106,shape,"shape, source, quality, format, themes",0.5265,0.0,0.0,0.0,0.3521,0.3500
6,7,The Godfather Part II,1974,240,0.4063,quality,"shape, quality, format, themes",0.2785,0.0,0.0,0.0,0.0000,1.0000
7,8,Chinatown,1974,829,0.4028,quality,"shape, quality, format, themes",0.2810,0.0,0.0,0.0,0.0000,0.9040
8,9,Se7en,1995,807,0.4027,shape,"shape, quality, format, themes",0.5987,0.0,0.0,0.0,0.0000,0.4000
9,10,The Prestige,2006,1124,0.3500,shape,"shape, director, quality, format, themes",0.2512,1.0,0.0,0.0,0.0000,0.3500
